In [31]:
import sys
from pathlib import Path
from pyprojroot import here

sys.path.append(str(here()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import cfg
from src.my_utils import set_seed, cv_result, log_experiment, make_submit

from sklearn.model_selection import RepeatedStratifiedKFold

from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

from src.features import *
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer


In [32]:
%load_ext autoreload
%autoreload 2


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [17]:
gseed = cfg.general.seed
set_seed(gseed)


In [18]:
train_path = Path(cfg.paths.train)
submit_path = Path(cfg.paths.test)


In [19]:
df_train = pd.read_csv(train_path)
df_submit = pd.read_csv(submit_path)


In [20]:
X_train = df_train.drop(columns=["Survived"])
y_train = df_train["Survived"]


In [21]:
rskf = RepeatedStratifiedKFold(n_splits=10, n_repeats=5, random_state=gseed)

In [22]:
preprocessor = Pipeline(
    [
        ("dtypes", DtypesTransformer()),
        ("title_transformer", TitleTransformer()),
        ("age_imputer", ByPclassAgeImputer()),
        ("embarked_imputer", EmbarkedImputer()),
        ("fare_imputer", FareImputer()),
        ("group_transformer", GroupTransformer(smooth=1e-6)),
        ("cabin_transformer", AdvancedCabinTransformer()),
        (
            "column_transformer",
            ColumnTransformer(
                [
                    (
                        "cat_ohe",
                        OneHotEncoder(sparse_output=False, drop="first"),
                        ["Pclass", "Sex", "Title", "Has_Cabin"],
                    ),
                    (
                        "num",
                        "passthrough",
                        ["Age", "Ticket_Group_Size", "Fare", "Ticket_Survival_Rate"],
                    ),
                ],
                remainder="drop",
                verbose_feature_names_out=False,
            ).set_output(transform="pandas"),
        ),
    ]
)

### consistent model

In [23]:
base = RandomForestClassifier(
    n_estimators=300, max_depth=5, min_samples_split=10, random_state=gseed
)

In [24]:
_, _, metrics = cv_result(base, X_train, y_train, rskf, preprocessor)
metrics

,TRAIN_acc_MEAN,TRAIN_acc_STD,VAL_acc_MEAN,VAL_acc_STD,OOF_acc
0,0.890311,0.006919,0.840849,0.036838,0.845118


In [ ]:
make_submit(base, preprocessor, X_train, y_train, df_submit, "rfc_final_features.csv")

Файл успешно сохранен: d:\vs_projects\fp_titanic\data\external\rfc_final_features.csv


### baseline lgbm

In [26]:
baseline_model = LGBMClassifier(random_state=gseed, max_depth=7)

In [27]:
_, _, metrics = cv_result(baseline_model, X_train, y_train, rskf, preprocessor)
metrics

[LightGBM] [Info] Number of positive: 307, number of negative: 494
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000291 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 229
[LightGBM] [Info] Number of data points in the train set: 801, number of used features: 12
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.383271 -> initscore=-0.475688
[LightGBM] [Info] Start training from score -0.475688
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best

,TRAIN_acc_MEAN,TRAIN_acc_STD,VAL_acc_MEAN,VAL_acc_STD,OOF_acc
0,0.942462,0.008384,0.843106,0.036359,0.842873


In [ ]:
make_submit(
    base, preprocessor, X_train, y_train, df_submit, "bl_no_tune_final_features.csv"
)

Файл успешно сохранен: d:\vs_projects\fp_titanic\data\external\bl_no_tune_final_features.csv


### no tune catboost

In [ ]:
baseline_model = CatBoostClassifier(random_seed=gseed)


In [51]:
_, _, metrics = cv_result(baseline_model, X_train, y_train, rskf, preprocessor)
metrics


Learning rate set to 0.009371
0:	learn: 0.6861129	total: 1.61ms	remaining: 1.61s
1:	learn: 0.6790595	total: 3.14ms	remaining: 1.57s
2:	learn: 0.6728121	total: 4.46ms	remaining: 1.48s
3:	learn: 0.6661493	total: 5.59ms	remaining: 1.39s
4:	learn: 0.6596805	total: 6.7ms	remaining: 1.33s
5:	learn: 0.6532724	total: 7.83ms	remaining: 1.3s
6:	learn: 0.6476014	total: 8.9ms	remaining: 1.26s
7:	learn: 0.6423734	total: 9.96ms	remaining: 1.23s
8:	learn: 0.6365185	total: 11ms	remaining: 1.21s
9:	learn: 0.6309801	total: 12.1ms	remaining: 1.2s
10:	learn: 0.6255269	total: 13.3ms	remaining: 1.19s
11:	learn: 0.6205036	total: 14.1ms	remaining: 1.16s
12:	learn: 0.6152816	total: 15.1ms	remaining: 1.15s
13:	learn: 0.6099207	total: 16.2ms	remaining: 1.14s
14:	learn: 0.6057070	total: 17.7ms	remaining: 1.16s
15:	learn: 0.6009370	total: 19.1ms	remaining: 1.18s
16:	learn: 0.5970302	total: 20.4ms	remaining: 1.18s
17:	learn: 0.5925956	total: 21.5ms	remaining: 1.17s
18:	learn: 0.5872791	total: 22.6ms	remaining: 1.17

,TRAIN_acc_MEAN,TRAIN_acc_STD,VAL_acc_MEAN,VAL_acc_STD,OOF_acc
0,0.927023,0.005692,0.846245,0.031439,0.855219


In [35]:
make_submit(
    base, preprocessor, X_train, y_train, df_submit, "cb_no_tune_final_features.csv"
)

Файл успешно сохранен: d:\vs_projects\fp_titanic\data\external\cb_no_tune_final_features.csv


### tuning lgbm

In [52]:
import optuna
from optuna.samplers import TPESampler

In [53]:
def objective_lgbm(trial):
    params = {
        "random_state": gseed,
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "num_leaves": trial.suggest_int("num_leaves", 15, 255),
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.3, log=True),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
    }

    model = LGBMClassifier(**params, verbosity=-1)
    _, _, metrics = cv_result(model, X_train, y_train, rskf, preprocessor)

    return metrics["OOF_acc"].item()

In [55]:
study_lgbm = optuna.create_study(direction="maximize", sampler=TPESampler(seed=gseed))
study_lgbm.optimize(objective_lgbm, n_trials=100, show_progress_bar=True)

[I 2026-09-05 19:28:44,267] A new study created in memory with name: no-name-8d0f0b76-e653-4381-baf8-5949ee5153b4


  0%|          | 0/100 [00:00<?, ?it/s]

[I 2026-09-05 19:28:49,353] Trial 0 finished with value: 0.8462401795735129 and parameters: {'n_estimators': 565, 'max_depth': 7, 'num_leaves': 21, 'learning_rate': 0.010091633082984282, 'min_child_samples': 70, 'subsample': 0.9169484313180383, 'colsample_bytree': 0.6534831098361189, 'reg_alpha': 1.1028532241013205, 'reg_lambda': 0.03118133864355715}. Best is trial 0 with value: 0.8462401795735129.
[I 2026-09-05 19:28:53,396] Trial 1 finished with value: 0.8451178451178452 and parameters: {'n_estimators': 271, 'max_depth': 7, 'num_leaves': 99, 'learning_rate': 0.010529365650093677, 'min_child_samples': 80, 'subsample': 0.9827416112059846, 'colsample_bytree': 0.6161768309073803, 'reg_alpha': 5.650057960050972e-08, 'reg_lambda': 0.0027035586822681547}. Best is trial 0 with value: 0.8462401795735129.
[I 2026-09-05 19:28:58,154] Trial 2 finished with value: 0.8406285072951739 and parameters: {'n_estimators': 756, 'max_depth': 5, 'num_leaves': 180, 'learning_rate': 0.041669347616674714, 'mi

In [56]:
print(study_lgbm.best_params)
print(study_lgbm.best_value)

{'n_estimators': 409, 'max_depth': 5, 'num_leaves': 104, 'learning_rate': 0.04755571719306031, 'min_child_samples': 21, 'subsample': 0.7279664239579899, 'colsample_bytree': 0.5322640057787851, 'reg_alpha': 0.1057163625351844, 'reg_lambda': 4.628797701922884}
0.8574635241301908


In [60]:
best_lgbm = LGBMClassifier(**study_lgbm.best_params, random_state=gseed, verbosity=-1)
_, _, metrics = cv_result(best_lgbm, X_train, y_train, rskf, preprocessor)
metrics

,TRAIN_acc_MEAN,TRAIN_acc_STD,VAL_acc_MEAN,VAL_acc_STD,OOF_acc
0,0.927722,0.006749,0.848931,0.034657,0.857464


In [61]:
make_submit(
    best_lgbm, preprocessor, X_train, y_train, df_submit, "best_lgbm_final_features.csv"
)

Файл успешно сохранен: d:\vs_projects\fp_titanic\data\external\best_lgbm_final_features.csv


### tuning catboost

In [57]:
def objective_catboost(trial):
    params = {
        "random_state": gseed,
        "iterations": trial.suggest_int("iterations", 200, 1500),
        "depth": trial.suggest_int("depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.3, log=True),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 10.0, log=True),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 10.0),
        "border_count": trial.suggest_int("border_count", 32, 255),
        "verbose": False,
    }

    model = CatBoostClassifier(**params)

    _, _, metrics = cv_result(model, X_train, y_train, rskf, preprocessor)

    return metrics["OOF_acc"].item()

In [58]:
study_cb = optuna.create_study(direction="maximize", sampler=TPESampler(seed=gseed))
study_cb.optimize(objective_catboost, n_trials=100, show_progress_bar=True)

[I 2026-09-05 19:39:50,108] A new study created in memory with name: no-name-a37fa381-51b2-4dc6-a67f-7461cf906204


  0%|          | 0/100 [00:00<?, ?it/s]

[I 2026-09-05 19:41:11,245] Trial 0 finished with value: 0.8540965207631874 and parameters: {'iterations': 871, 'depth': 7, 'learning_rate': 0.0056182555017538546, 'l2_leaf_reg': 1.4842998932803027, 'bagging_temperature': 6.852769816973126, 'border_count': 218}. Best is trial 0 with value: 0.8540965207631874.
[I 2026-09-05 19:43:48,603] Trial 1 finished with value: 0.8383838383838383 and parameters: {'iterations': 599, 'depth': 10, 'learning_rate': 0.09593655612243439, 'l2_leaf_reg': 1.5485989276883412, 'bagging_temperature': 5.542275911247872, 'border_count': 110}. Best is trial 0 with value: 0.8540965207631874.
[I 2026-09-05 19:45:15,901] Trial 2 finished with value: 0.8249158249158249 and parameters: {'iterations': 436, 'depth': 9, 'learning_rate': 0.2604624992542696, 'l2_leaf_reg': 1.7074722798750672, 'bagging_temperature': 0.8356143366334368, 'border_count': 167}. Best is trial 0 with value: 0.8540965207631874.
[I 2026-09-05 19:46:26,146] Trial 3 finished with value: 0.82267115600

In [59]:
print(study_cb.best_params)
print(study_cb.best_value)

{'iterations': 760, 'depth': 7, 'learning_rate': 0.009990468903622285, 'l2_leaf_reg': 1.3717265878688303, 'bagging_temperature': 3.583425469372962, 'border_count': 206}
0.8574635241301908


In [64]:
best_cb = CatBoostClassifier(**study_cb.best_params, random_state=gseed, verbose=0)
_, _, metrics = cv_result(best_cb, X_train, y_train, rskf, preprocessor)
metrics

,TRAIN_acc_MEAN,TRAIN_acc_STD,VAL_acc_MEAN,VAL_acc_STD,OOF_acc
0,0.929667,0.005789,0.846697,0.031033,0.857464


In [65]:
make_submit(
    best_cb, preprocessor, X_train, y_train, df_submit, "best_cb_final_features.csv"
)

Файл успешно сохранен: d:\vs_projects\fp_titanic\data\external\best_cb_final_features.csv


# Лучше всего себя показал LGBM, его скор лидерборда составил 0.78468 > 0.77751 (у catboost)